# RawGrad — L2 Regularization

This experiment demonstrates L2 regularization in RawGrad.

We will:

- understand why regularization is needed
- train a baseline model without regularization
- train a model with L2 regularization
- measure the L2 penalty
- verify that regularization gradients are applied
- compare the two models
- validate the complete pipeline

The implementation is built from scratch using NumPy.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.model import Sequential
from rawgrad.classification_losses import CategoricalCrossEntropy
from rawgrad.optimizers import SGD

In [2]:
np.random.seed(42)

X_train = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 0.5],
    [0.0, 1.0, 2.0],
    [3.0, 2.0, 1.0],
    [1.5, 2.5, 3.0],
    [2.5, 1.0, 0.5],
    [0.2, 1.2, 2.2],
    [3.2, 2.0, 1.0],
])

y_train = np.array([
    0, 1, 2, 1,
    0, 1, 2, 1
])

X_test = np.array([
    [1.2, 2.1, 2.8],
    [2.2, 1.1, 0.4],
    [0.1, 1.1, 2.1],
    [3.1, 1.9, 1.1],
])

y_test = np.array([0, 1, 2, 1])

num_classes = 3

Y_train = np.zeros((len(y_train), num_classes))
Y_train[np.arange(len(y_train)), y_train] = 1.0

Y_test = np.zeros((len(y_test), num_classes))
Y_test[np.arange(len(y_test)), y_test] = 1.0

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Number of classes:", num_classes)

Training samples: 8
Testing samples: 4
Number of classes: 3


In [3]:
def create_model(l2_lambda=0.0):
    return Sequential([
        Dense(
            3,
            8,
            initialization="xavier",
            l2_lambda=l2_lambda,
        ),
        ReLU(),
        Dense(
            8,
            3,
            initialization="xavier",
            l2_lambda=l2_lambda,
        ),
        Softmax(),
    ])

In [4]:
np.random.seed(42)

baseline_model = create_model(l2_lambda=0.0)
regularized_model = create_model(l2_lambda=0.1)

baseline_penalty = baseline_model.regularization_loss()
regularized_penalty = regularized_model.regularization_loss()

print("Baseline L2 penalty:   ", baseline_penalty)
print("Regularized L2 penalty:", regularized_penalty)

assert baseline_penalty == 0.0
assert regularized_penalty > 0.0

print("L2 penalty calculation: PASS")

Baseline L2 penalty:    0.0
Regularized L2 penalty: 0.3680823885868309
L2 penalty calculation: PASS


In [5]:
np.random.seed(42)

model = create_model(l2_lambda=0.1)

predictions = model.forward(X_train)

loss_function = CategoricalCrossEntropy()

loss = loss_function.forward(
    predictions,
    Y_train,
)

gradient = loss_function.backward()

model.backward(gradient)

dense_layers = [
    layer
    for layer in model.layers
    if isinstance(layer, Dense)
]

for layer in dense_layers:
    assert layer.weights_gradient.shape == layer.weights.shape
    assert layer.biases_gradient.shape == layer.biases.shape
    assert np.all(np.isfinite(layer.weights_gradient))
    assert np.all(np.isfinite(layer.biases_gradient))

print("Regularization gradients: PASS")

Regularization gradients: PASS


In [6]:
def train_model(model, epochs=300):
    loss_function = CategoricalCrossEntropy()
    optimizer = SGD(learning_rate=0.1)

    data_loss_history = []
    penalty_history = []
    objective_history = []

    for epoch in range(epochs):
        predictions = model.forward(X_train)

        data_loss = loss_function.forward(
            predictions,
            Y_train,
        )

        gradient = loss_function.backward()

        model.backward(gradient)

        optimizer.step(model)

        penalty = model.regularization_loss()
        objective = data_loss + penalty

        data_loss_history.append(data_loss)
        penalty_history.append(penalty)
        objective_history.append(objective)

    return (
        data_loss_history,
        penalty_history,
        objective_history,
    )

In [7]:
np.random.seed(42)

baseline_model = create_model(l2_lambda=0.0)

(
    baseline_data_loss,
    baseline_penalty,
    baseline_objective,
) = train_model(
    baseline_model,
    epochs=300,
)

print("Baseline training completed.")
print("Initial data loss:", baseline_data_loss[0])
print("Final data loss:  ", baseline_data_loss[-1])
print("Final L2 penalty: ", baseline_penalty[-1])
print("Final objective:  ", baseline_objective[-1])

Baseline training completed.
Initial data loss: 0.7890724264340684
Final data loss:   0.044177138325425004
Final L2 penalty:  0.0
Final objective:   0.044177138325425004


In [8]:
np.random.seed(42)

regularized_model = create_model(l2_lambda=0.1)

(
    regularized_data_loss,
    regularized_penalty,
    regularized_objective,
) = train_model(
    regularized_model,
    epochs=300,
)

print("Regularized training completed.")
print("Initial data loss:", regularized_data_loss[0])
print("Final data loss:  ", regularized_data_loss[-1])
print("Final L2 penalty: ", regularized_penalty[-1])
print("Final objective:  ", regularized_objective[-1])

Regularized training completed.
Initial data loss: 0.7890724264340684
Final data loss:   0.22819618097495134
Final L2 penalty:  0.2492341359748424
Final objective:   0.47743031694979376


In [9]:
print("REGULARIZATION COMPARISON")
print("-" * 50)

print(
    f"Baseline    | "
    f"Data Loss: {baseline_data_loss[-1]:.6f} | "
    f"L2 Penalty: {baseline_penalty[-1]:.6f} | "
    f"Objective: {baseline_objective[-1]:.6f}"
)

print(
    f"Regularized | "
    f"Data Loss: {regularized_data_loss[-1]:.6f} | "
    f"L2 Penalty: {regularized_penalty[-1]:.6f} | "
    f"Objective: {regularized_objective[-1]:.6f}"
)

REGULARIZATION COMPARISON
--------------------------------------------------
Baseline    | Data Loss: 0.044177 | L2 Penalty: 0.000000 | Objective: 0.044177
Regularized | Data Loss: 0.228196 | L2 Penalty: 0.249234 | Objective: 0.477430


In [10]:
loss_function = CategoricalCrossEntropy()

baseline_predictions = baseline_model.forward(X_test)

baseline_test_loss = loss_function.forward(
    baseline_predictions,
    Y_test,
)

baseline_test_classes = np.argmax(
    baseline_predictions,
    axis=1,
)

baseline_test_accuracy = np.mean(
    baseline_test_classes == y_test
)

regularized_predictions = regularized_model.forward(X_test)

regularized_test_loss = loss_function.forward(
    regularized_predictions,
    Y_test,
)

regularized_test_classes = np.argmax(
    regularized_predictions,
    axis=1,
)

regularized_test_accuracy = np.mean(
    regularized_test_classes == y_test
)

print("TEST EVALUATION")
print("-" * 50)

print(
    f"Baseline    | "
    f"Loss: {baseline_test_loss:.6f} | "
    f"Accuracy: {baseline_test_accuracy * 100:.2f}%"
)

print(
    f"Regularized | "
    f"Loss: {regularized_test_loss:.6f} | "
    f"Accuracy: {regularized_test_accuracy * 100:.2f}%"
)

TEST EVALUATION
--------------------------------------------------
Baseline    | Loss: 0.035887 | Accuracy: 100.00%
Regularized | Loss: 0.239476 | Accuracy: 100.00%


In [11]:
def total_weight_squared_sum(model):
    total = 0.0

    for layer in model.layers:
        if isinstance(layer, Dense):
            total += np.sum(layer.weights ** 2)

    return total


baseline_weight_magnitude = total_weight_squared_sum(
    baseline_model
)

regularized_weight_magnitude = total_weight_squared_sum(
    regularized_model
)

print("WEIGHT MAGNITUDE")
print("-" * 40)

print(
    f"Baseline    : {baseline_weight_magnitude:.6f}"
)

print(
    f"Regularized : {regularized_weight_magnitude:.6f}"
)

WEIGHT MAGNITUDE
----------------------------------------
Baseline    : 19.879183
Regularized : 4.984683


In [12]:
assert np.all(np.isfinite(baseline_data_loss))
assert np.all(np.isfinite(regularized_data_loss))

assert np.all(np.isfinite(baseline_objective))
assert np.all(np.isfinite(regularized_objective))

assert regularized_penalty[-1] > 0.0

assert baseline_predictions.shape == (4, 3)
assert regularized_predictions.shape == (4, 3)

assert np.all(np.isfinite(baseline_predictions))
assert np.all(np.isfinite(regularized_predictions))

assert np.allclose(
    np.sum(baseline_predictions, axis=1),
    1.0,
)

assert np.allclose(
    np.sum(regularized_predictions, axis=1),
    1.0,
)

assert baseline_test_accuracy >= 0.0
assert regularized_test_accuracy >= 0.0

print("REGULARIZATION: PASS")

REGULARIZATION: PASS
